In [33]:
import numpy as np
import torch 
import torch.nn as nn 
import torch.nn.functional as F
import matplotlib.pyplot as plt 

In [34]:
import pandas as pd
iris = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv')

data = torch.tensor( iris[iris.columns[0:4]].values ).float()

labels = torch.zeros(len(data), dtype=torch.long)
labels[iris.species=='versicolor'] = 1
labels[iris.species=='virginica'] = 2

In [35]:
propTraining = .8 
nTraining = int(len(labels)*propTraining)

traintestBool = np.zeros(len(labels),dtype=bool)

items2use4train = np.random.choice(range(len(labels)),nTraining,replace=False)
traintestBool[items2use4train] = True

traintestBool


array([False,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True, False,
        True,  True,  True,  True,  True,  True,  True, False,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True, False,  True,  True,
        True,  True,  True,  True, False,  True, False, False,  True,
        True,  True,  True,  True, False,  True,  True, False,  True,
        True, False,  True,  True,  True,  True, False,  True,  True,
       False,  True,  True, False,  True, False,  True,  True,  True,
       False, False,  True,  True,  True, False, False, False, False,
        True,  True,  True, False,  True,  True,  True,  True, False,
       False,  True,

In [36]:

# test whether it's balanced
print('Average of full data:')
print( torch.mean(labels.float()) ) # =1 by definition
print(' ')

print('Average of training data:')
print( torch.mean(labels[traintestBool].float()) ) # should be 1...
print(' ')

print('Average of test data:')
print( torch.mean(labels[~traintestBool].float()) ) # should also be 1...

Average of full data:
tensor(1.)
 
Average of training data:
tensor(0.8917)
 
Average of test data:
tensor(1.4333)


In [37]:
ANNiris = nn.Sequential(
    nn.Linear(4,64),   # input layer
    nn.ReLU(),         # activation unit
    nn.Linear(64,64),  # hidden layer
    nn.ReLU(),         # activation unit
    nn.Linear(64,3),   # output units
      )

lossfun = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(ANNiris.parameters(),lr=.01)
     

In [38]:
numepochs = 1000

losses = torch.zeros(numepochs)
ongoingAcc = []

for epochi in range(numepochs):

  yHat = ANNiris(data[traintestBool,:])

  ongoingAcc.append( 100*torch.mean(
              (torch.argmax(yHat,axis=1) == labels[traintestBool]).float()) )

  loss = lossfun(yHat,labels[traintestBool])
  losses[epochi] = loss

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
     

In [39]:
predictions = ANNiris(data[traintestBool,:])
trainacc = 100*torch.mean((torch.argmax(predictions,axis=1) == labels[traintestBool]).float())

predictions = ANNiris(data[~traintestBool,:])
testacc = 100*torch.mean((torch.argmax(predictions,axis=1) == labels[~traintestBool]).float())

perClassAcc = {int(c): (100*torch.mean((torch.argmax(predictions,axis=1)[labels[~traintestBool]==c] == labels[~traintestBool][labels[~traintestBool]==c]).float())).item() for c in torch.unique(labels)}
perClassAcc
     

{0: 100.0, 1: 100.0, 2: 88.23529815673828}

In [40]:
print('Final TRAIN accuracy: %g%%' %trainacc)
print('Final TEST accuracy:  %g%%' %testacc)
print(perClassAcc)

Final TRAIN accuracy: 98.3333%
Final TEST accuracy:  93.3333%
{0: 100.0, 1: 100.0, 2: 88.23529815673828}
